# Data Extraction

In [112]:
import obspy, io, subprocess, utils, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# import obspy.core.utcdatetime as utc
# from obspy.clients.fdsn import Client
# from eqcct.stream import st_predictor
# from eqcct import plot_traces

WIN_SIZE = 60               # Window size in seconds. Same window size used in EQCCT paper
PRE_EVENT_BUFFER = 5        # Number of seconds from start of window until first pick
POST_EVENT_BUFFER = 5       # Number of seconds from last pick until end of window

ISUH_IP_ADDR = "http://128.214.169.201:8080"
WAVEFORM_DIR = "waveforms"
PRED_DIR = "predictions"

## Opening large mseed files
`obspy.read()` cannot directly read mini-SEED files that are larger than 2048 MB. Read the file in chunks instead ([link](https://github.com/obspy/obspy/pull/1419#issuecomment-221582369) to github issue).

Note on `record_len`: A record is a unit of data in the miniSEED format. A record consists of a certain number of data points in the waveform. Typical record lengths for raw data generation and archiving are between 256 and 4096 bytes ([FDSN docs](https://docs.fdsn.org/projects/miniseed3/en/latest/definition.html#:~:text=typical%20record%20lengths%20for%20raw%20data%20generation%20and%20archiving%20are%20recommended%20to%20be%20in%20the%20range%20of%20256%20and%204096%20bytes)).

In [3]:
mseed_filename = "HE_020125_030125.mseed" # Queried from ISUH FDSNWS. It is a 3.7GB file containing waveforms from HE network from 2-3 Jan 2025
data_dir = Path("data")

record_len = 512
chunksize = 100000 * record_len
with io.open(data_dir / mseed_filename, "rb") as fh:
    while True:
        with io.BytesIO() as buf:
            c = fh.read(chunksize)
            if not c:
                break
            buf.write(c)
            buf.seek(0, 0)
            st = obspy.read(buf)
        # Do something useful!
        print(st)
        break # We just want to see the first stream from the mseed file

2 Trace(s) in Stream:
HE.OBF8..HHE | 2025-01-01T23:59:59.568000Z - 2025-01-03T00:00:00.748000Z | 250.0 Hz, 21600296 samples
HE.OBF8..HHN | 2025-01-01T23:59:58.808000Z - 2025-01-02T07:26:55.188000Z | 250.0 Hz, 6704096 samples


## Querying waveform data from ISUH FDSNWS

To query ISUH FDSNWS dataselect service, ensure your device is connected to the university's network. Eduroam does not work, but wired connections on devices at the university do.

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime


client = Client("http://128.214.169.201:8080") # ISUH FDSNWS IP address

start_time = UTCDateTime("2025-01-01T23:59:59.568000Z")
end_time = UTCDateTime("2025-01-03T00:00:00.748000Z")

st = client.get_waveforms("HE", "OBF8", "*", "HHZ", start_time, end_time)

st.plot()

## Creating waveform dataset

#### 1. Extract pick information from quakeML files into `df_pick`

In [93]:
data_dir = Path("data")

paths = {
    "earthquakes": data_dir / "earthquakes2025.qkml",
    "explosions": data_dir / "explosions2025.qkml",
    "probable_explosions": data_dir / "probable_explosions2025.qkml",
}

catalogs = {}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path.resolve()}")
    catalogs[name] = obspy.read_events(str(path))
    print(f"{name}: {len(catalogs[name])} events loaded from {path.name}")

# Dataframe of all earthquake, explosion and probable_explosion picks               
df_picks = utils.extract_picks(catalogs)
# df_picks.to_csv("picks.csv") # Export to csv for possible debugging

print(f"Total number of picks: {len(df_picks)}")
print(df_picks["Event type"].value_counts().to_frame(name="Number of picks"))
df_picks.head()

earthquakes: 328 events loaded from earthquakes2025.qkml
explosions: 599 events loaded from explosions2025.qkml
probable_explosions: 1455 events loaded from probable_explosions2025.qkml
Total number of picks: 54134
                     Number of picks
Event type                          
probable_explosions            32720
explosions                     14281
earthquakes                     7133


,Event id,SEED string,Pick type,Event type,Pick time
0,smi:fi.isuh/event/1490639,UP.LANU..BHZ,P,earthquakes,2025-01-01T09:22:01.256470Z
1,smi:fi.isuh/event/1490639,UP.LANU..BHZ,S,earthquakes,2025-01-01T09:22:07.069410Z
2,smi:fi.isuh/event/1490639,HE.HEF..BHZ,P,earthquakes,2025-01-01T09:22:02.651570Z
3,smi:fi.isuh/event/1490639,HE.HEF..BHZ,S,earthquakes,2025-01-01T09:22:09.394590Z
4,smi:fi.isuh/event/1490639,FN.KLF..BHZ,P,earthquakes,2025-01-01T09:22:07.766970Z


#### 2. Identify windows to query waveforms from ISUH FDSNWS

Note about events:
* Each seismic event has zero or more P picks, and zero or more S picks
* Multiple stations can produce picks for the same event
* Therefore, a window will be uniquely identified by its event-station tuple (`Event id`, `SEED string short`)

Window definition:
* Each window should contain one P pick and one S pick, similar to the STEAD dataset used in benchmarking EQCCT. The STEAD dataset uses 60s windows.
* But if P and S arrival times may be too far from each other, each pick will be put in its own separate query

In [ ]:
# Select earthquake picks
df_picks_eq = df_picks[df_picks["Event type"] == "earthquakes"]

# Remove channel ID from the SEED string, so that the shortened SEED string can identify stations
df_picks_eq["SEED string short"] = df_picks_eq["SEED string"].str.rsplit(".", n=1).str[0] 


# Report how many P and S picks are made at each event-station 
num_PS_picks = []
for (event_id, seed_string), df_group in df_picks_eq.groupby(["Event id", "SEED string short"]):
    num_P_picks = int(df_group["Pick type"].str.startswith("P").sum())
    num_S_picks = int(df_group["Pick type"].str.startswith("S").sum())
    num_PS_picks.append((num_P_picks, num_S_picks))
print(pd.Series(num_PS_picks, name="(num P picks, num S picks)").value_counts().to_frame("Number of event-stations").reset_index())
print("")


# Time of the earliest pick (usually P) for an event-station
df_picks_eq["Event start"] = df_picks_eq.groupby(["Event id", df_picks_eq["SEED string short"]])["Pick time"].transform("min")

# Time of the latest pick (usually S) for an event-station
df_picks_eq["Event end"] = df_picks_eq.groupby(["Event id", df_picks_eq["SEED string short"]])["Pick time"].transform("max")

# Ensure that query window starts 5s before the first pick (usually P)
df_picks_eq["Win start"] = df_picks_eq["Event start"] - PRE_EVENT_BUFFER

# Ensure that query window is WIN_SIZE seconds long
df_picks_eq["Win end"] = df_picks_eq["Win start"] + WIN_SIZE


# Mask for P-S pairs that are too far (far pairs) to be contained by the window; Window too short to contain P, S, 5s buffer after S wave, and 5s buffer before P wave
mask_far_picks = df_picks_eq["Win end"] - POST_EVENT_BUFFER < df_picks_eq["Event end"]


# Report the number of far pairs
df_far_picks = df_picks_eq[mask_far_picks]
df_far_picks["P-S time diff (s)"] = df_far_picks["Event end"] - df_far_picks["Event start"] # Extract time difference between P and S picks for inspection

max_diff = max(df_picks_eq["Event end"] - df_picks_eq["Event start"])               # Max time diff between P-S pair
num_far_pairs = len(df_far_picks.value_counts(["Event id", "SEED string short"]))   # Number of far pairs

print(f"Max duration between P and S arrivals: {max_diff}s\n")
print(f"The window size is too short to contain P, S picks, and buffers for these events ({num_far_pairs} far pairs, {len(df_far_picks)} far picks):")


# Proceed to put each far picks into their own separate windows
df_picks_eq.loc[mask_far_picks, "Win start"] = df_picks_eq.loc[mask_far_picks, "Pick time"] - PRE_EVENT_BUFFER
df_picks_eq.loc[mask_far_picks, "Win end"] = df_picks_eq.loc[mask_far_picks, "Win start"] + WIN_SIZE
df_picks_eq.loc[mask_far_picks, "Event id"] = df_picks_eq.loc[mask_far_picks, "Event id"] + df_picks_eq.loc[mask_far_picks, "Pick type"] # Each window will be identified by event-station fields later, but we need to distinguish between windows of far picks since they have their own separate windows, so we add suffix for differentiation


# df_picks_eq.to_csv("eq_picks.csv", index=False)  # Save picks to csv for debugging

df_far_picks

  (num P picks, num S picks)  Number of event-stations
0                     (1, 1)                      2844
1                     (0, 1)                      1369
2                     (1, 0)                        46
3                     (1, 2)                         5
4                     (2, 1)                         3
5                     (0, 2)                         3

Max duration between P and S arrivals: 86.70201s

The window size is too short to contain P, S picks, and buffers for these events (16 far pairs, 33 far picks):


,Event id,SEED string,Pick type,Event type,Pick time,SEED string short,Event start,Event end,Win start,Win end,P-S time diff (s)
256,smi:fi.isuh/event/1491766,NS.STOK..?HZ,P,earthquakes,2025-01-12T20:07:08.896530Z,NS.STOK.,2025-01-12T20:07:08.896530Z,2025-01-12T20:08:05.293930Z,2025-01-12T20:07:03.896530Z,2025-01-12T20:08:03.896530Z,56.3974
257,smi:fi.isuh/event/1491766,NS.STOK..?HZ,S,earthquakes,2025-01-12T20:08:05.293930Z,NS.STOK.,2025-01-12T20:07:08.896530Z,2025-01-12T20:08:05.293930Z,2025-01-12T20:07:03.896530Z,2025-01-12T20:08:03.896530Z,56.3974
258,smi:fi.isuh/event/1491766,UP.HEMU..BHZ,P,earthquakes,2025-01-12T20:07:20.381200Z,UP.HEMU.,2025-01-12T20:07:20.381200Z,2025-01-12T20:08:26.692320Z,2025-01-12T20:07:15.381200Z,2025-01-12T20:08:15.381200Z,66.31112
259,smi:fi.isuh/event/1491766,UP.HEMU..BHZ,S,earthquakes,2025-01-12T20:08:26.692320Z,UP.HEMU.,2025-01-12T20:07:20.381200Z,2025-01-12T20:08:26.692320Z,2025-01-12T20:07:15.381200Z,2025-01-12T20:08:15.381200Z,66.31112
260,smi:fi.isuh/event/1491766,UP.HUSU..BHZ,P,earthquakes,2025-01-12T20:07:27.662460Z,UP.HUSU.,2025-01-12T20:07:27.662460Z,2025-01-12T20:08:39.484710Z,2025-01-12T20:07:22.662460Z,2025-01-12T20:08:22.662460Z,71.82225
261,smi:fi.isuh/event/1491766,UP.HUSU..BHZ,S,earthquakes,2025-01-12T20:08:39.484710Z,UP.HUSU.,2025-01-12T20:07:27.662460Z,2025-01-12T20:08:39.484710Z,2025-01-12T20:07:22.662460Z,2025-01-12T20:08:22.662460Z,71.82225
262,smi:fi.isuh/event/1491766,UP.NRTU..BHZ,P,earthquakes,2025-01-12T20:07:35.279830Z,UP.NRTU.,2025-01-12T20:07:35.279830Z,2025-01-12T20:08:51.486290Z,2025-01-12T20:07:30.279830Z,2025-01-12T20:08:30.279830Z,76.20646
263,smi:fi.isuh/event/1491766,UP.NRTU..BHZ,S,earthquakes,2025-01-12T20:08:51.486290Z,UP.NRTU.,2025-01-12T20:07:35.279830Z,2025-01-12T20:08:51.486290Z,2025-01-12T20:07:30.279830Z,2025-01-12T20:08:30.279830Z,76.20646
264,smi:fi.isuh/event/1491766,HE.ECKF..CZ,P,earthquakes,2025-01-12T20:07:37.052100Z,HE.ECKF.,2025-01-12T20:07:37.052100Z,2025-01-12T20:08:53.717110Z,2025-01-12T20:07:32.052100Z,2025-01-12T20:08:32.052100Z,76.66501
265,smi:fi.isuh/event/1491766,HE.ECKF..CZ,S,earthquakes,2025-01-12T20:08:53.717110Z,HE.ECKF.,2025-01-12T20:07:37.052100Z,2025-01-12T20:08:53.717110Z,2025-01-12T20:07:32.052100Z,2025-01-12T20:08:32.052100Z,76.66501


#### 3. Generating labels for waveforms

The labels are stored in a CSV file, with the same columns as EQCCTOne's prediction output csv file. 

In [ ]:
# Column names copied from EQCCTOne's prediction output csv file
COLUMNS_Y = ["file_name", "network", "station", "instrument_type", "station_lat", "station_lon", "station_elv","p_arrival_time", "p_probability", "s_arrival_time", "s_probability"]

df_y = pd.DataFrame(columns=COLUMNS_Y)

# Generate labels for each window. Each window has either both P and S times, or only one of them
for (event_id, seed_string), df_group in df_picks_eq.groupby(["Event id", "SEED string"]):
    win_start = df_group["Win start"].iloc[0]
    win_end = df_group["Win end"].iloc[0]
    
    p_time, s_time = None, None
    # If the event-station has P picks, choose the arrival time of the first P pick
    if df_group["Pick type"].str.startswith("P").any(): 
        p_time = df_group[df_group["Pick type"].str.startswith("P")]["Pick time"].iloc[0] 
    else:
        p_time = np.nan
    
    # If the event-station has S picks, choose the arrival time of the first S pick
    if df_group["Pick type"].str.startswith("S").any():
        s_time = df_group[df_group["Pick type"].str.startswith("S")]["Pick time"].iloc[0] 
    else:
        s_time = np.nan
    
    seed_string = seed_string.replace("..", ".00.") # Fill in missing fields in SEED string with 00 to match eqcct's prediction output format
    
    file_name = seed_string + " | " + str(win_start) + " - " + str(win_end)
    row_df = pd.DataFrame([[file_name, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 
                            p_time, np.nan, s_time, np.nan]], columns=COLUMNS_Y)
    
    df_y = pd.concat([df_y, row_df])

# df_y.to_csv("eq_labels.csv", index=False) # Save picks to csv for debugging
df_y

,file_name,network,station,instrument_type,station_lat,station_lon,station_elv,p_arrival_time,p_probability,s_arrival_time,s_probability
0,FN.KLF.00.BHZ | 2025-01-01T09:22:02.766970Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01T09:22:07.766970Z,NaN,2025-01-01T09:22:17.997750Z,NaN
0,FN.RNF.00.BHZ | 2025-01-01T09:22:18.810700Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01T09:22:23.810700Z,NaN,2025-01-01T09:22:45.899900Z,NaN
0,FN.SGF.00.BHZ | 2025-01-01T09:22:12.997750Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01T09:22:17.997750Z,NaN,2025-01-01T09:22:36.134150Z,NaN
0,HE.HEF.00.BHZ | 2025-01-01T09:21:57.651570Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01T09:22:02.651570Z,NaN,2025-01-01T09:22:09.394590Z,NaN
0,NO.ARA0.00.?HZ | 2025-01-01T09:22:42.760040Z -...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01T09:22:47.760040Z,NaN
...,...,...,...,...,...,...,...,...,...,...,...
0,HE.KY17.00.CZ | 2025-11-10T01:45:35.562920Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-10T01:45:40.562920Z,NaN,2025-11-10T01:45:47.332480Z,NaN
0,HE.LOVF.00.CZ | 2025-11-10T01:45:32.088540Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-10T01:45:37.088540Z,NaN,2025-11-10T01:45:41.443310Z,NaN
0,HE.PVF.00.CZ | 2025-11-10T01:45:30.303730Z - 2...,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-10T01:45:35.303730Z,NaN,2025-11-10T01:45:38.293170Z,NaN
0,HE.VJF.00.?HZ | 2025-11-10T01:45:40.102750Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-10T01:45:45.102750Z,NaN,2025-11-10T01:45:55.778810Z,NaN


#### 4. Generating Predictions 
There are two options:

1. Query ISUH FDSNWS for mseed files to store locally OR
2. Query windows then output predictions iteratively without storing mseed files

In [102]:
# Generate windows to query, identified based on event id and station id (SEED string short)
df_windows = df_picks_eq.drop_duplicates(["Event id", "SEED string short"]) 

df_windows.to_csv("eq_windows.csv", index=False) # Save windows to CSV to run on eqcct in another virtual environment lol

##### 4a. Option 1: Query ISUH FDSNWS for mseed files

In [ ]:
# Use create_dataset.py utility from eqcctpro
for seed_string, start, end in zip(df_windows["SEED string short"], df_windows["Win start"], df_windows["Win end"]):
    print(seed_string)
    # create_dataset.py can only be used via command line
    cmd = ["python", "create_dataset.py", 
                    "--start", str(start), "--end", str(end), 
                    "--streams", seed_string,
                    "--host", ISUH_IP_ADDR,
                    "--output", WAVEFORM_DIR,
                    "--chunk", "1"] # chunk size (minutes)
    subprocess.run(cmd)

UP.LANU..
Total time: 2025-01-01T09:21:56.256470Z → 2025-01-01T09:22:07.069410Z  (10.81294 s)
Chunk size: 2 minute(s) → 1 window(s)


=== WINDOW 1/1: 20250101T092156Z → 20250101T092207Z ===
  Fetching for: UP.LANU.*.*
HTTP Status code: 204
Detailed response of server:


  No data downloaded for this window.

All windows processed. Done.
HE.HEF..
Total time: 2025-01-01T09:21:57.651570Z → 2025-01-01T09:22:09.394590Z  (11.74302 s)
Chunk size: 2 minute(s) → 1 window(s)


=== WINDOW 1/1: 20250101T092157Z → 20250101T092209Z ===
  Fetching for: HE.HEF.*.*
  Wrote /home/ad/lxhome/n/ngdeqi/Linux/Documents/DATA11004 Project/eqdet/waveforms_directory/20250101T092157Z_20250101T092209Z/HEF/HE.HEF..HHE__20250101T092157Z__20250101T092209Z.mseed  (1 traces)
  Wrote /home/ad/lxhome/n/ngdeqi/Linux/Documents/DATA11004 Project/eqdet/waveforms_directory/20250101T092157Z_20250101T092209Z/HEF/HE.HEF..HHN__20250101T092157Z__20250101T092209Z.mseed  (1 traces)
  Wrote /home/ad/lxhome/n/ngdeqi/Linux/Documents/DA

Traceback (most recent call last):
  File "/home/ad/lxhome/n/ngdeqi/Linux/Documents/DATA11004 Project/eqdet/create_dataset.py", line 7, in <module>
    from obspy.clients.fdsn import Client
  File "/home/ngdeqi/Documents/DATA11004 Project/eqdet/.venv/lib/python3.12/site-packages/obspy/clients/fdsn/__init__.py", line 246, in <module>
    from .routing.routing_client import RoutingClient  # NOQA
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ngdeqi/Documents/DATA11004 Project/eqdet/.venv/lib/python3.12/site-packages/obspy/clients/fdsn/routing/routing_client.py", line 25, in <module>
    from ...base import HTTPClient
  File "/home/ngdeqi/Documents/DATA11004 Project/eqdet/.venv/lib/python3.12/site-packages/obspy/clients/base.py", line 55, in <module>
    import requests
  File "/home/ngdeqi/Documents/DATA11004 Project/eqdet/.venv/lib/python3.12/site-packages/requests/__init__.py", line 43, in <module>
    import urllib3
  File "/home/ngdeqi/Documents/DATA11004 Project

KeyboardInterrupt: 

#### 4b. Option 2: Query and predict without storing

In [ ]:
result_dir = "results"

# Generate windows to query, identified based on event id and station id (SEED string short)
df_windows = df_picks_eq.drop_duplicates(["Event id", "SEED string short"]) 

cl = Client(ISUH_IP_ADDR)

df_predictions = pd.DataFrame(columns=COLUMNS_Y)

for event_id, seed_string_s, win_start, win_end in zip(df_windows["Event id"], df_windows["SEED string short"], df_windows["Win start"], df_windows["Win end"]):

    network_code = seed_string_s.split(".")[0]
    station_code = seed_string_s.split(".")[1]
    location_code = "00"
    stream_code = "*" # Wildcard to select all streams
    
    # Query waveform from ISUH FDSNWS
    st = cl.get_waveforms(network_code, station_code, location_code, stream_code, win_start, win_end)
    
    try:
        st_predictor(input_modelP=r'eqcct\eqcctone\ModelPS\test_trainer_024.h5',
            input_modelS=r'eqcct\eqcctone\ModelPS\test_trainer_021.h5',
            stinput = st,
            output_dir=WAVEFORM_DIR,
            P_threshold=0.1,
            S_threshold=0.1, 
            number_of_plots=10,
            normalization_mode='std',
            batch_size=1,
            gpuid=None,
            gpu_limit=None,
            overwrite=True)
    except: 
        print(f"Prediction failed for (event id: {event_id}, SEED string short: {seed_string_s})")
        continue
    
    df_prediction = pd.read_csv(Path(WAVEFORM_DIR) / station_code / "X_prediction_results.csv")
    df_predictions = pd.concat([df_predictions, df_prediction])
    
    break

df_predictions.to_csv(Path(WAVEFORM_DIR) / "X_prediction_results.csv")

# data = pd.read_csv('detections_ALPN/ALPN_outputs/X_prediction_results.csv', low_memory=False)
# data.p_arrival_time

# pat=[ii for ii in list(data.p_arrival_time) if type(ii) is str]; #P arrival time
# sat=[ii for ii in list(data.s_arrival_time) if type(ii) is str]; #S arrival time

# plot_traces(st,axoff=1,titleoff=1,ptime= [obspy.UTCDateTime(pat[0]),obspy.UTCDateTime(pat[0]),obspy.UTCDateTime(pat[0])],stime=[obspy.UTCDateTime(sat[0]),obspy.UTCDateTime(sat[0]),obspy.UTCDateTime(sat[0])],figname='test_eqcct_texnet2022yplg.png',dpi=500)


# TODO:
1. Generate noise windows and labels